<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Solution: Readable queries

**[C04-M03-L01 · Readable queries](https://learning.nextia-ai.com/courses/sql/m03/readable-queries/)** · C04, SQL and Data Preparation for AI · Module 3, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** split a long SQL query into named steps with common table expressions (`WITH`), count the rows of each step, and find a quiet mistake before it reaches a report.

| | |
|---|---|
| **Time** | About 45 minutes |
| **Needs** | An internet connection for the first cell. No account and no install: only Python's standard library, and pandas to show tables if it is there (it is in Colab and Kaggle). |
| **You should know** | `SELECT`, `WHERE`, `GROUP BY`, `CASE` and `LEFT JOIN`, and how to count rows after a join, from [Aggregate correctly](https://learning.nextia-ai.com/courses/sql/m02/aggregate-correctly/) and [Join tables](https://learning.nextia-ai.com/courses/sql/m02/join-tables/). |
| **You do not need** | The local project from Module 1. The setup below builds the same database here. |
| **Tested** | 2026-10-07, Python 3.14.6 (SQLite 3.50.4), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m03/readable-queries/) has the full explanations, the pipeline diagram and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your query between the triple quotes `"""`, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M03-L01-readable-queries/readable-queries-solution.ipynb).

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One query, many jobs

Larkfield is a made-up online shop for home and garden products. Mei, the data engineer, prepares its help-desk export for a model. Omar, the product manager, asks her: "Which customer segment has the most escalated tickets?" One query must do four jobs:

1. Remove the 14 repeated ticket rows.
2. Find the tickets that have an `escalated` event in `outcomes`.
3. Join each ticket to its customer, to get the segment.
4. Count the tickets and the escalations per segment.

With nested subqueries, the middle results have no names, and you cannot count them. A **common table expression (CTE)** gives a middle result a name: `WITH name AS (query)` at the start of a query. The name exists only while that query runs. The main `SELECT` can then use it as a table.

### A tiny example

The next cell makes a tiny table in memory, `tiny`, with five ticket rows. `T-10002` is in it twice, an exact copy, as in the real export. Run the cell. It prints nothing.

In [ ]:
tiny = sqlite3.connect(":memory:")
tiny.executescript("""
CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, team TEXT);
INSERT INTO tickets VALUES
    ('T-10001', 'C-0001', 'delivery'),
    ('T-10002', 'C-0001', 'returns'),
    ('T-10002', 'C-0001', 'returns'),
    ('T-10003', 'C-0002', 'delivery'),
    ('T-10004', NULL, 'account');
""");

The CTE `unique_tickets` keeps one copy of each row. The main query counts its rows.

> **Predict.** How many rows does `unique_tickets` have? Write it down, then run the cell.

In [ ]:
sql("""
WITH unique_tickets AS (
    SELECT DISTINCT *
    FROM tickets
)
SELECT COUNT(*) AS row_count
FROM unique_tickets;
""", db=tiny)

> **Check.** You should see **4**.

> **Your turn: change one thing.** In the cell above, change the last line to `FROM tickets;`, and run it again. You should see **5**: the repeated row counts twice.

Two things make a CTE useful:

- **The step has a name.** `unique_tickets` says what the step does.
- **You can look inside the step.** Change only the final `SELECT`, and you see any step: its rows, or its row count.

You can define several CTEs after one `WITH`, with a comma between them. A later step can use an earlier one.

## 2. Worked example: escalations by segment

Mei writes the four jobs as four CTEs. The [lesson page](https://learning.nextia-ai.com/courses/sql/m03/readable-queries/#the-pipeline) has a diagram of the steps with their row counts. Comments name each step's job.

- Step 3 uses `CASE` to give every ticket a segment: `guest` when the ticket has no `customer_id`, `unknown` when the customer is not in `customers` (a deleted account), and the lower-case segment otherwise. `LOWER` makes `Trade` and `trade` one group.
- The second `CASE` turns "has a matching escalated row" into 1 and "no match" into 0. `SUM(escalated)` then counts the escalated tickets.

> **Predict.** The report has one row per segment. How many rows? Then run the cell.

In [ ]:
sql("""
-- Escalations by customer segment, in named steps.
WITH unique_tickets AS (
    -- Step 1: one row per ticket (the export has 14 repeated rows)
    SELECT DISTINCT *
    FROM tickets
),
escalated_tickets AS (
    -- Step 2: one row per ticket that has an escalated event
    SELECT DISTINCT ticket_id
    FROM outcomes
    WHERE status = 'escalated'
),
ticket_rows AS (
    -- Step 3: each ticket with its segment and a 0/1 escalated flag
    SELECT
        u.ticket_id,
        CASE
            WHEN u.customer_id IS NULL THEN 'guest'
            WHEN c.customer_id IS NULL THEN 'unknown'
            ELSE LOWER(c.segment)
        END AS segment,
        CASE WHEN e.ticket_id IS NULL THEN 0 ELSE 1 END AS escalated
    FROM unique_tickets AS u
    LEFT JOIN customers AS c
        ON c.customer_id = u.customer_id
    LEFT JOIN escalated_tickets AS e
        ON e.ticket_id = u.ticket_id
),
by_segment AS (
    -- Step 4: one row per segment
    SELECT
        segment,
        COUNT(*) AS tickets,
        SUM(escalated) AS escalated,
        ROUND(100.0 * SUM(escalated) / COUNT(*), 1) AS escalated_pct
    FROM ticket_rows
    GROUP BY segment
)
SELECT segment, tickets, escalated, escalated_pct
FROM by_segment
ORDER BY tickets DESC;
""")

> **Check.** You should see 5 rows: `trade` 634 tickets and 107 escalated (16.9%), `home` 403 and 44 (10.9%), `business` 345 and 70 (20.3%), `guest` 12 and 1 (8.3%), `unknown` 9 and 3 (33.3%).

Before she sends it, Mei checks that the totals match the source. Run the cell: it adds up the two columns of the report.

In [ ]:
columns, rows = run("""
WITH unique_tickets AS (
    -- Step 1: one row per ticket (the export has 14 repeated rows)
    SELECT DISTINCT *
    FROM tickets
),
escalated_tickets AS (
    -- Step 2: one row per ticket that has an escalated event
    SELECT DISTINCT ticket_id
    FROM outcomes
    WHERE status = 'escalated'
),
ticket_rows AS (
    -- Step 3: each ticket with its segment and a 0/1 escalated flag
    SELECT
        u.ticket_id,
        CASE
            WHEN u.customer_id IS NULL THEN 'guest'
            WHEN c.customer_id IS NULL THEN 'unknown'
            ELSE LOWER(c.segment)
        END AS segment,
        CASE WHEN e.ticket_id IS NULL THEN 0 ELSE 1 END AS escalated
    FROM unique_tickets AS u
    LEFT JOIN customers AS c
        ON c.customer_id = u.customer_id
    LEFT JOIN escalated_tickets AS e
        ON e.ticket_id = u.ticket_id
),
by_segment AS (
    -- Step 4: one row per segment
    SELECT
        segment,
        COUNT(*) AS tickets,
        SUM(escalated) AS escalated,
        ROUND(100.0 * SUM(escalated) / COUNT(*), 1) AS escalated_pct
    FROM ticket_rows
    GROUP BY segment
)
SELECT segment, tickets, escalated, escalated_pct
FROM by_segment
ORDER BY tickets DESC;
""")
print("tickets:  ", sum(row[1] for row in rows))
print("escalated:", sum(row[2] for row in rows))

> **Check.** You should see **1403** tickets, the number of distinct tickets, and **225** escalated, the number of rows in `escalated_tickets`.

Her answer to Omar: business customers escalate most often (20.3%), and trade customers have the most escalated tickets (107). The guest and unknown groups are too small to compare.

## 3. Look inside each step

The final `SELECT` can read any CTE. To check a step, replace only the last part of the query. Run the cell as it is: it shows the row count of `ticket_rows`.

> **Predict.** `ticket_rows` starts from `unique_tickets` (1,403 rows) and adds two LEFT JOINs. How many rows must it have?

In [ ]:
sql("""
WITH unique_tickets AS (
    -- Step 1: one row per ticket (the export has 14 repeated rows)
    SELECT DISTINCT *
    FROM tickets
),
escalated_tickets AS (
    -- Step 2: one row per ticket that has an escalated event
    SELECT DISTINCT ticket_id
    FROM outcomes
    WHERE status = 'escalated'
),
ticket_rows AS (
    -- Step 3: each ticket with its segment and a 0/1 escalated flag
    SELECT
        u.ticket_id,
        CASE
            WHEN u.customer_id IS NULL THEN 'guest'
            WHEN c.customer_id IS NULL THEN 'unknown'
            ELSE LOWER(c.segment)
        END AS segment,
        CASE WHEN e.ticket_id IS NULL THEN 0 ELSE 1 END AS escalated
    FROM unique_tickets AS u
    LEFT JOIN customers AS c
        ON c.customer_id = u.customer_id
    LEFT JOIN escalated_tickets AS e
        ON e.ticket_id = u.ticket_id
)
SELECT COUNT(*) AS row_count
FROM ticket_rows;
""")

> **Check.** You should see **1403**. A LEFT JOIN keeps every left row. When each right side has at most one row per key, the join adds columns but no rows. So this count must not change.

> **Your turn: change one thing.** In the cell above, change `FROM ticket_rows` to `FROM escalated_tickets`, and run it: 225. Then replace the last two lines with `SELECT * FROM ticket_rows WHERE segment = 'unknown';` to see the 9 tickets of deleted customers.

## 4. Guided practice: the step counts

Mei saves a second query next to the report. It prints every step's row count at once. `UNION ALL` puts the rows of several queries one under the other. Each `SELECT` must have the same number of columns.

On your computer, save the report as `queries/31-segment-report.sql` and this query as `queries/32-step-counts.sql`, as on the lesson page. Here, run the cell.

In [ ]:
sql("""
-- Row count of each step in 31-segment-report.sql.
WITH unique_tickets AS (
    -- Step 1: one row per ticket (the export has 14 repeated rows)
    SELECT DISTINCT *
    FROM tickets
),
escalated_tickets AS (
    -- Step 2: one row per ticket that has an escalated event
    SELECT DISTINCT ticket_id
    FROM outcomes
    WHERE status = 'escalated'
),
ticket_rows AS (
    -- Step 3: each ticket with its segment and a 0/1 escalated flag
    SELECT
        u.ticket_id,
        CASE
            WHEN u.customer_id IS NULL THEN 'guest'
            WHEN c.customer_id IS NULL THEN 'unknown'
            ELSE LOWER(c.segment)
        END AS segment,
        CASE WHEN e.ticket_id IS NULL THEN 0 ELSE 1 END AS escalated
    FROM unique_tickets AS u
    LEFT JOIN customers AS c
        ON c.customer_id = u.customer_id
    LEFT JOIN escalated_tickets AS e
        ON e.ticket_id = u.ticket_id
)
SELECT 'tickets' AS step, COUNT(*) AS row_count FROM tickets
UNION ALL
SELECT 'unique_tickets', COUNT(*) FROM unique_tickets
UNION ALL
SELECT 'escalated_tickets', COUNT(*) FROM escalated_tickets
UNION ALL
SELECT 'ticket_rows', COUNT(*) FROM ticket_rows
UNION ALL
SELECT 'distinct tickets in ticket_rows', COUNT(DISTINCT ticket_id) FROM ticket_rows;
""")

> **Check.** You should see 5 rows: `tickets` 1417, `unique_tickets` 1403, `escalated_tickets` 225, `ticket_rows` 1403, `distinct tickets in ticket_rows` 1403. The last two must be equal: one row per ticket in `ticket_rows`.

## 5. Your turn: escalations by team

Grace, the support lead, asks the same question per team. Change one condition: group by `team` instead of the customer's segment. The team is a column of `tickets`, so you do not need the `customers` table.

> **Your turn.** Add a step `ticket_rows` with `ticket_id`, `team` and a 0/1 `escalated` flag. Then return three columns: `team`, `tickets`, `escalated`. The result has 5 rows. The tickets must add up to 1,403 and the escalations to 225. Then run the check cell. The row order does not count.

In [ ]:
by_team = """
WITH unique_tickets AS (
    SELECT DISTINCT *
    FROM tickets
),
escalated_tickets AS (
    SELECT DISTINCT ticket_id
    FROM outcomes
    WHERE status = 'escalated'
),
ticket_rows AS (
    SELECT
        u.ticket_id,
        u.team,
        CASE WHEN e.ticket_id IS NULL THEN 0 ELSE 1 END AS escalated
    FROM unique_tickets AS u
    LEFT JOIN escalated_tickets AS e
        ON e.ticket_id = u.ticket_id
)
SELECT team, COUNT(*) AS tickets, SUM(escalated) AS escalated
FROM ticket_rows
GROUP BY team
ORDER BY tickets DESC;
"""
sql(by_team)

In [ ]:
check(by_team, rows=5, columns=3, expected="631e77d291f682f2")

Warranty tickets escalate most often. The team counts are a little smaller than in the raw table (delivery: 449, not 452), because `unique_tickets` removed the repeated rows.

## 6. Failure case: a step without DISTINCT adds three tickets

Tomás copies the report and removes `DISTINCT` from `escalated_tickets`, because "a ticket is escalated or not". The query runs with no error.

> **Predict.** Which step count changes? Then run the cell. It is the step-count query with Tomás's step 2.

In [ ]:
sql("""
WITH unique_tickets AS (
    -- Step 1: one row per ticket (the export has 14 repeated rows)
    SELECT DISTINCT *
    FROM tickets
),
escalated_tickets AS (
    -- Step 2 WITHOUT DISTINCT (Tomás's version)
    SELECT ticket_id
    FROM outcomes
    WHERE status = 'escalated'
),
ticket_rows AS (
    -- Step 3: each ticket with its segment and a 0/1 escalated flag
    SELECT
        u.ticket_id,
        CASE
            WHEN u.customer_id IS NULL THEN 'guest'
            WHEN c.customer_id IS NULL THEN 'unknown'
            ELSE LOWER(c.segment)
        END AS segment,
        CASE WHEN e.ticket_id IS NULL THEN 0 ELSE 1 END AS escalated
    FROM unique_tickets AS u
    LEFT JOIN customers AS c
        ON c.customer_id = u.customer_id
    LEFT JOIN escalated_tickets AS e
        ON e.ticket_id = u.ticket_id
)
SELECT 'tickets' AS step, COUNT(*) AS row_count FROM tickets
UNION ALL
SELECT 'unique_tickets', COUNT(*) FROM unique_tickets
UNION ALL
SELECT 'escalated_tickets', COUNT(*) FROM escalated_tickets
UNION ALL
SELECT 'ticket_rows', COUNT(*) FROM ticket_rows
UNION ALL
SELECT 'distinct tickets in ticket_rows', COUNT(DISTINCT ticket_id) FROM ticket_rows;
""")

> **Check.** You should see `escalated_tickets` **228** and `ticket_rows` **1406**, but `distinct tickets in ticket_rows` still **1403**. The first count that goes up is the step with the problem.

**Cause:** `outcomes` has one row per event, not per ticket. Three tickets have their escalated event logged twice (a retry: T-10115, T-10887 and T-11005). The LEFT JOIN matches each of them twice. The segment report then adds up to 1,406 tickets and 228 escalations, and nothing warns you.

**Fix:** make each step have the grain that the next step needs: one row per ticket, so `SELECT DISTINCT ticket_id`. Then run the counts again.

> **Warning.** In a training table, a ticket that appears twice gets more weight, and it can land in both the train set and the test set. Only a row count shows it.

## Check your understanding and recap

Answer the three **knowledge checks** at the end of the [lesson page](https://learning.nextia-ai.com/courses/sql/m03/readable-queries/). They count toward your certificate when you are signed in and enrolled.

In this lesson you learned that a CTE (`WITH name AS (...)`) gives a middle result a name, and that several CTEs make a pipeline of steps. To check a step, change only the final `SELECT`. Write down the row count that each step must have, save a step-count query next to each report, and run it after every change.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Time and ranking](https://learning.nextia-ai.com/courses/sql/m03/time-and-ranking/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m03/readable-queries/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.